# RPS-1 Strategy Analysis

**Course:** MSE 3302B - Sensors and Actuators  
**Team:** MSE-08 (Evan Romano, Joseph Toma, Mohammed Alamen Qassab)  
**Date:** March 2026

This notebook documents how we developed the strategy used by our RPS-1 system during live matches. We started by deriving a Nash-equilibrium baseline so our Stage 1 choices could not be exploited easily, then added a lightweight adaptive layer based on opponent tendencies observed during play. This document explains how we arrived at the control logic before coding it into the final firmware.

## 1. Game Model and Decision Structure

In Rock-Paper-Scissors Minus One, each round had two connected decisions. At Stage 1, each side showed a pair of gestures, and at Stage 2 each side kept one of the two. To keep the analysis manageable, we represented Stage 1 using the canonical pairs `RP`, `RS`, and `PS`, since these captured the meaningful non-duplicate options and matched the way we tracked rounds in code through `getComboIndex()` and `comboCount[3]`.

For Stage 2, we treated the keep decision as a direct payoff comparison between our two shown hands and the two opponent hands. This is the same structure implemented in `pickStage2()` in `FullGameComplete.ino`, where both candidates are scored against both opponent options and the stronger keep is selected.

## 2. Nash Equilibrium Baseline

We used a mixed strategy at Stage 1 because fixed pair choices would become predictable over repeated rounds. Let the probabilities of playing `RP`, `RS`, and `PS` be `x`, `y`, and `z`, with `x + y + z = 1`. At equilibrium, the opponent should not gain by favoring one response over another, so we applied the indifference condition across the canonical options. This produced the ratio

$$x:y:z = 2:1:2$$

and after normalization the baseline became

$$p_{RP}=0.4,\quad p_{RS}=0.2,\quad p_{PS}=0.4.$$

This distribution was exactly what we implemented in `pickNashCombo()`. The function used `random(5)`, mapped outcomes `{0,1}` to `RP`, `{2}` to `RS`, and `{3,4}` to `PS`, which gave the same 2/5, 1/5, and 2/5 probabilities from the derivation.

## 3. Stage 2 Keep Rule

After both players showed their two-hand combinations, we selected which hand to keep by comparing expected safety against both opponent options. If our hands were `(g1, g2)` and opponent hands were `(o1, o2)`, we calculated

$$s_1=u(g_1,o_1)+u(g_1,o_2),\quad s_2=u(g_2,o_1)+u(g_2,o_2),$$

with RPS payoff values `win = +1`, `tie = 0`, and `loss = -1`. We kept `g1` when `s1 >= s2`, otherwise we kept `g2`. This was the same scoring rule used in `pickStage2()`, and we kept the tie-break fixed so behavior stayed consistent during fast rounds.

## 4. Adaptive Layer and Firmware Thresholds

Nash gave us a strong baseline, but in actual class matches we observed that many opponents repeated the same Stage 1 pair or showed clear keep tendencies. Because of that, we added a small adaptive layer based on direct frequency counts instead of a heavy statistical model. The firmware tracked opponent pair frequency in `comboCount[3]`, keep tendencies in `comboKeep[3][2]`, and total completed rounds in `totalRounds`.

The exact trigger values in `pickStage1()` were kept simple. We used pure Nash while `totalRounds < 6`. After that point, we found the most frequent opponent pair index `mf`, and only switched to counter mode when

$$\frac{\texttt{comboCount[mf]}}{\texttt{totalRounds}} > 0.5.$$

If this condition was not met, we stayed on Nash. That kept early rounds stable and only enabled adaptation when opponent bias was clear.

## 5. Counter Selection Logic

When the adaptive trigger fired, we used `predictAndCounter()` to move away from neutral sampling and target the opponent's dominant pattern. The function first estimated which hand the opponent was more likely to keep inside their dominant combo, using `comboKeep`, and then selected our next Stage 1 pair to pressure that likely keep.

In practice, this approach gave us an interpretable strategy layer that was still light enough for real-time tournament use. It also avoided abrupt behavior changes, because activation depended on both a minimum number of rounds and a dominance threshold greater than 0.5.

## 6. Final Strategy Definition Used for Control

The control logic we finalized combined a game-theoretic baseline and a constrained adaptive extension. Stage 1 sampling followed the Nash distribution `(RP, RS, PS) = (0.4, 0.2, 0.4)`. Stage 2 used the payoff comparison rule from `pickStage2()` to select the keep hand. Adaptation was gated by firmware thresholds (`totalRounds >= 6` and dominant pair share `> 0.5`) and then executed through `predictAndCounter()` using observed pair and keep histories.

In class matches, this gave us steady baseline behavior against unknown opponents while still letting the system capitalize on repeated patterns.